# Pepper-image denoising

Compare all six ToMoBAR denoisers exposed by `httomolibgpu.misc.denoise` on the
same reproducible noisy image. Adapted from ToMoBAR's `demos/denoising/peppers.py`;
the bundled `tests/test_data/peppers.tif` is copied from ToMoBAR.

Run from the repository root on a CUDA GPU:

```bash
python -m pip install -e '.[validation]'
python -m pytest --nbmake --nbmake-timeout=1800 --overwrite docs/source/validation/denoising/denoising_peppers.ipynb
```

TV/TGV settings are unchanged. Diffusion settings are tuned to this image and
noise realisation, rather than an independent test dataset.
PSNR and SSIM use the clean reference and unmodified outputs; display limits
only affect the figures. Documentation renders saved outputs without execution.


In [ ]:
import gc
from pathlib import Path

import cupy as cp
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

from httomolibgpu.misc.denoise import (
    total_variation_ROF,
    total_variation_PD,
    total_variation_SB,
    total_variation_FGP,
    total_generalised_variation,
    nonlinear_diffusion,
)

repo_root = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "tests/test_data/peppers.tif").is_file()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        "Run from the repository or a subdirectory; tests/test_data/peppers.tif is required"
    )
with Image.open(repo_root / "tests/test_data/peppers.tif") as image:
    clean = np.asarray(image.convert("L"), dtype=np.float32) / np.float32(255.0)
rng = np.random.default_rng(42)
noise_sigma = 0.15
noisy = np.ascontiguousarray(
    clean + rng.normal(0, noise_sigma, clean.shape).astype(np.float32)
)
# The wrappers accept 3D volumes; a singleton Z axis represents this 2D image.
noisy_gpu = cp.asarray(noisy[None])
print(f"Image: {clean.shape}, Gaussian noise sigma: {noise_sigma}, seed: 42")
print(f"CuPy {cp.__version__}; GPU: {cp.cuda.runtime.getDeviceProperties(0)['name']}")

## Denoising

All methods start from the same unmodified input. TV/TGV use full-precision
work arrays here. NDF supports only full precision; its five edge-stopping
penalties are shown separately. Equal strengths do not imply equal smoothing
across methods. The noisy Gaussian values are not clipped before processing.

The diffusion settings below are the best balanced candidates from **3,387 GPU
evaluations** over strength, sigma, iteration count and time step. Selection
maximises `PSNR + 20 * SSIM` among candidates that improve both metrics over
the original settings; 0.01 SSIM therefore has the weight of 0.2 dB PSNR.
This is a search optimum for this fixed reference/noise pair, not a global
optimum or a general-purpose default. The selected iteration counts include
early stopping; ToMoBAR may cap the requested time step internally.

[Search results](../../_static/validation/peppers_diffusion_search.csv) ·
[Selected settings and provenance](../../_static/validation/peppers_diffusion_selected.json).


In [ ]:
denoisers = [
    (
        "ROF-TV",
        total_variation_ROF,
        dict(
            regularisation_parameter=0.15,
            iterations=2000,
            time_marching_parameter=0.005,
        ),
    ),
    (
        "PD-TV",
        total_variation_PD,
        dict(regularisation_parameter=0.15, iterations=500, nonnegativity=True),
    ),
    (
        "SB-TV",
        total_variation_SB,
        dict(regularisation_parameter=0.15, iterations=300, nonnegativity=True),
    ),
    (
        "FGP-TV",
        total_variation_FGP,
        dict(regularisation_parameter=0.15, iterations=300, nonnegativity=True),
    ),
    (
        "TGV",
        total_generalised_variation,
        dict(
            regularisation_parameter=0.15,
            alpha1=1.0,
            alpha0=1.65,
            iterations=1500,
            nonnegativity=True,
        ),
    ),
]
diffusion_settings = [
    (
        "NDF Huber",
        dict(
            regularisation_parameter=0.24,
            sigma=0.018,
            iterations=64,
            time_marching_parameter=0.1,
            penalty_type=1,
            nonnegativity=True,
        ),
    ),
    (
        "NDF Perona-Malik",
        dict(
            regularisation_parameter=8.0,
            sigma=0.1,
            iterations=40,
            time_marching_parameter=0.1,
            penalty_type=2,
            nonnegativity=True,
        ),
    ),
    (
        "NDF Tukey",
        dict(
            regularisation_parameter=24.0,
            sigma=0.9,
            iterations=8,
            time_marching_parameter=0.1,
            penalty_type=3,
            nonnegativity=True,
        ),
    ),
    (
        "NDF truncated quadratic",
        dict(
            regularisation_parameter=7.2,
            sigma=0.84,
            iterations=12,
            time_marching_parameter=0.025,
            penalty_type=4,
            nonnegativity=True,
        ),
    ),
    (
        "NDF truncated Huber",
        dict(
            regularisation_parameter=2.016,
            sigma=0.34,
            iterations=9,
            time_marching_parameter=0.025,
            penalty_type=5,
            nonnegativity=True,
        ),
    ),
]
for name, parameters in diffusion_settings:
    denoisers.append((name, nonlinear_diffusion, parameters))

results = {}
for name, denoise, parameters in denoisers:
    filtered = denoise(noisy_gpu, gpu_id=0, half_precision=False, **parameters)
    assert filtered.shape == noisy_gpu.shape, name
    assert filtered.dtype == cp.float32 and filtered.flags.c_contiguous, name
    assert bool(cp.isfinite(filtered).all()), name
    results[name] = cp.asnumpy(filtered[0])
    del filtered
np.testing.assert_array_equal(noisy_gpu.get()[0], noisy)

## Quality and images

Compare PSNR (higher is better) and SSIM (closer to one is better) against the
clean image. All images share the same display range. Metrics describe this
image and noise realisation; they are not universal method rankings.


In [ ]:
quality = {}
print(f"{'Image':26s} {'PSNR (dB)':>10s} {'SSIM':>10s}")
for name, result in [("Noisy", noisy), *results.items()]:
    psnr = peak_signal_noise_ratio(clean, result, data_range=1.0)
    ssim = structural_similarity(clean, result, data_range=1.0)
    quality[name] = (psnr, ssim)
    print(f"{name:26s} {psnr:10.3f} {ssim:10.4f}")

fig, axes = plt.subplots(3, 4, figsize=(16, 12), constrained_layout=True)
for ax, (name, result) in zip(
    axes.flat, [("Clean reference", clean), ("Noisy", noisy), *results.items()]
):
    ax.imshow(result, cmap="gray", vmin=0, vmax=1)
    title = name
    if name in quality:
        psnr, ssim = quality[name]
        title += f"\nPSNR {psnr:.2f} dB | SSIM {ssim:.3f}"
    ax.set_title(title)
    ax.set_axis_off()
plt.show()
plt.close(fig)

del noisy_gpu
# Keep the host-side images and metrics available for inspection.
gc.collect()
cp.get_default_memory_pool().free_all_blocks()